In [2]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Preview Datasets
from datasets import load_dataset

DATASETS = {
    "gsm8k": ("gsm8k", "main"),
    "asdiv": ("yimingzhang/asdiv", None),
    "metamathqa": ("meta-math/MetaMathQA", None),
    "openmathinstruct": ("nvidia/OpenMathInstruct-1", None), # Can change to nvidia/OpenMathInstruct-2 for larger datasets
}

for name, (path, subset) in DATASETS.items():
    print("\n" + "=" * 70)
    print(f"Dataset: {name}")
    print("=" * 70)

    try:
        # Load dataset
        if subset:
            dataset = load_dataset(path, subset)
        else:
            dataset = load_dataset(path)

        print("Available splits:", list(dataset.keys()))

        for split in dataset.keys():
            print(f"\n--- Split: {split} ---")
            print("Number of samples:", len(dataset[split]))
            
            # Column names
            print("Column names:", dataset[split].column_names)
            
            # Feature types (schema)
            print("Features:")
            print(dataset[split].features)

            # Print one example
            print("\nFirst sample:")
            print(dataset[split][0])

    except Exception as e:
        print("Error loading dataset:", e)


Dataset: gsm8k
Available splits: ['train', 'test']

--- Split: train ---
Number of samples: 7473
Column names: ['question', 'answer']
Features:
{'question': Value('string'), 'answer': Value('string')}

First sample:
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

--- Split: test ---
Number of samples: 1319
Column names: ['question', 'answer']
Features:
{'question': Value('string'), 'answer': Value('string')}

First sample:
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer'

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\Lew Chin Yee\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lew Chin Yee\.cache\huggingface\hub\datasets--nvidia--OpenMathInstruct-1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to a

Available splits: ['train', 'validation']

--- Split: train ---
Number of samples: 7321344
Column names: ['question', 'expected_answer', 'predicted_answer', 'error_message', 'is_correct', 'generation_type', 'dataset', 'generated_solution']
Features:
{'question': Value('string'), 'expected_answer': Value('string'), 'predicted_answer': Value('string'), 'error_message': Value('string'), 'is_correct': Value('bool'), 'generation_type': Value('string'), 'dataset': Value('string'), 'generated_solution': Value('string')}

First sample:
{'question': 'Martha has 18 crayons. She lost half of them, so she bought a new set of 20 crayons. How many crayons in total does Martha have after the purchase?', 'expected_answer': '29', 'predicted_answer': '29', 'error_message': '', 'is_correct': True, 'generation_type': 'masked_reference_solution', 'dataset': 'gsm8k', 'generated_solution': "Let's solve this problem using Python code.\n<llm-code>\namount_of_lost_crayons = 18 / 2\namount_of_new_crayons = 20\nt

In [ ]:
# Configure Datasets
DATASET_CONFIG = {
    "gsm8k": {
        "path": "gsm8k",
        "subset": "main",
        "split": "test",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["answer"]
    },
    "asdiv": {
        "path": "yimingzhang/asdiv",
        "subset": None,
        "split": "test",
        "extract_question": lambda x: x["text"].replace("Question:", "").replace("Answer:", "").strip(),
        "extract_answer": lambda x: x["label"]   # clean numeric answer
    },
    "metamathqa": {
        "path": "meta-math/MetaMathQA",
        "subset": None,
        "split": "train", # No testing dataset available
        "extract_question": lambda x: x["original_question"],
        "extract_answer": lambda x: x["response"]
    },
    "openmathinstruct": {
        "path": "nvidia/OpenMathInstruct-1", # Can change to nvidia/OpenMathInstruct-2 for larger datasets
        "subset": None,
        "split": "validation",
        "extract_question": lambda x: x["question"],
        "extract_answer": lambda x: x["expected_answer"]
    }
}

In [ ]:
# Build Prompt
def build_prompt(question, name):
    return f"""
You are a deterministic mathematics reasoning engine.

Solve the following math problem step by step using clear logical reasoning.
Show all intermediate reasoning clearly.
At the end, provide the final numerical answer.

Follow these steps:
1. Analyze the problem.
2. Compute step by step.
3. Double-check calculations.
4. Provide final result.

Important rules:
- Do NOT include any text outside the JSON.
- Do NOT use markdown.
- Ensure the final answer is a single number or expression.
- Do NOT restate the instructions.

Output your response strictly in the following JSON format:
{{
  "dataset": "{name}",
  "question": "...",
  "reasoning": "...",
  "final_answer": "..."
}}

Problem:
{question}
"""

In [ ]:
# Generate JSON Output
import json

all_outputs = []

for name, config in DATASET_CONFIG.items():

    print(f"Processing {name}...")

    dataset = load_dataset(
        config["path"],
        config["subset"] if config["subset"] else None
    )

    split_data = dataset[config["split"]]

    for sample in split_data:

        question = config["extract_question"](sample)
        answer = config.get("extract_answer", lambda x: None)(sample)        
        prompt = build_prompt(question, name)

        # ---- send to your model here ----
        # response_text = model.generate(prompt)

        # For demonstration only:
        response = {
            "dataset": name,
            "question": question,
            "reasoning": "model reasoning here",
            "final_answer": "model answer",
            "expencted_answer": answer
        }

        all_outputs.append(response)

        # Optional: limit samples per dataset
        if len(all_outputs) >= 100:
            break

# Save to JSON
with open("math_outputs.json", "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4, ensure_ascii=False)

Processing gsm8k...
Processing asdiv...
Processing metamathqa...
Processing openmathinstruct...
